In [1]:
import pandas as pd
import pyarrow.parquet as pq
import sys
import os
import numpy as np
import random as r


sys.path.insert(0, os.path.join(os.getcwd(), 'chronoepilogi_implementation'))

# Now import the class
from ce_extensions2 import ChronoEpilogi

In [6]:
db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
##Cleaning of all the row having a nan for a columns 
##One-Hot encoding 
print(db.columns.tolist())
print(db["Component"].dtype)

['CS_C2_TIT10_celcius', 'CS_C2_TIT20_celcius', 'CS_C2_PT20_bar', 'CS_C2_PT312_bar', 'CS_C2_PT322_bar', 'CS_C2_PT400_bar', 'CS_C2_PT60_bar', 'CS_C2_PT61_bar', 'CS_C1_TIT61_celcius', 'CS_C2_PT10_bar', 'AM_Temperature_celcius', 'CS_C1_PT10_bar', 'CS_C1_PT20_bar', 'CS_C1_PT312_bar', 'CS_C1_PT322_bar', 'CS_C1_PT400_bar', 'CS_C1_PT60_bar', 'CS_C1_PT61_bar', 'CS_C1_TIT10_celcius', 'CS_C1_TIT20_celcius', 'CS_C1_TIT21_celcius', 'CS_C1_TIT400_celcius', 'CS_C1_TIT60_celcius', 'ES_PA_Pressure_bar', 'DS_D2_Targeted_Pressure_bar', 'DS_D2_Room_Temperature_celcius', 'DS_D2_Requested_Pressure_bar', 'DS_D2_Nozzle_Temperature_celcius', 'DS_D2_Nozzle_Pressure_bar', 'DS_D2_Inlet_Pressure_bar', 'DS_D2_Fuel_Coupling_Temperature_celcius', 'DS_D2_Fuel_Coupling_Pressure_bar', 'DS_D1_Targeted_Pressure_bar', 'DS_D1_Room_Temperature_celcius', 'DS_D1_Requested_Pressure_bar', 'DS_D1_Nozzle_Temperature_celcius', 'DS_D1_Nozzle_Pressure_bar', 'DS_D1_Inlet_Pressure_bar', 'DS_D1_Fuel_Coupling_Temperature_celcius', 'DS_D1

In [7]:
db["Component"]

0         None
1         None
2         None
3         None
4         None
          ... 
218008    None
218009    None
218010    None
218011    None
218012    None
Name: Component, Length: 218013, dtype: object

In [68]:
#pd.set_option('display.max_columns', None)
#db.head()
column_names = db_encoded.columns.tolist()
columnsToDrop =  ["EPISODE_ID"] + [col for col in column_names if not col.startswith("Alarm_ID_") and db_encoded[col].dtypes != float]
print(columnsToDrop)

['EPISODE_ID', 'Date_Time', 'System', 'Alarm_State', 'Component', 'Class', 'Anomaly_Class']


In [ ]:
###ONE HOT ENCODING OF THE ALARMS

db = pd.read_parquet("RUL_FRMTM.parquet", engine="fastparquet")
##Cleaning of all the row having a nan for a columns 
##One-Hot encoding 
column_to_OHE="Alarm_ID"
db_encoded = pd.get_dummies(db, prefix="Alarm_ID", columns=[column_to_OHE], dtype=int)

#########################################################
# THIS CAN BE USED TO REDUCE THE DATASET SIZE IF NEEDED
non_na_rows = db[~db["System"].isna()]
na_rows = db[db["System"].isna()].sample(n=5000, random_state=42)  # Adjust 'n' as needed
db_sampled = pd.concat([non_na_rows, na_rows])
db_sampled.sort_values(by=["Date_Time", "EPISODE_ID"], inplace=True)
db=db_sampled.copy()
##########################################################

db_encoded = pd.get_dummies(db_sampled, prefix="Alarm_ID", columns=[column_to_OHE])
print(f"Size of the sampled database: {db_encoded.shape}")

##Dropping of EPISODE_ID and DateTime to not let them appear in the Markov Boundaries
db_encoded = db_encoded.drop(columns="EPISODE_ID")

target = 'RUL'  
target_type =  "continuous"
column_names = db_encoded.columns.tolist()


## we take all the columns that are of type float and the categorical one we want for the model
numerical_columns = [col for col in column_names if db_encoded[col].dtype == np.float64 and col != target]
categorical_columns = [col for col in column_names if col.startswith("Alarm_ID_")]

## and add them to our features database
features_db = numerical_columns + categorical_columns + [target]


#X = pd.concat([features_db, db_encoded[target]], axis=1)
X = db_encoded[features_db].copy()

## to prevent dtype issue, we set the type to a suitable one, just in case it hasnt done it by itself

X[numerical_columns] = X[numerical_columns].astype(np.float64)
X[categorical_columns] = X[categorical_columns].astype(np.int64)

## Data summary
print(f"Keeping {len(numerical_columns)+len(categorical_columns)+len(target)} columns:")
print(f"Categorical columns: {len(categorical_columns)}")
print(f"Numerical columns: {len(numerical_columns)}")
print(f"Target:{target}")


variable_types = {
    **{col: "numerical" for col in numerical_columns},
    **{col: "categorical" for col in categorical_columns}
}


Size of the sampled database: (6740, 181)
Keeping 176 columns:
Categorical columns: 88
Numerical columns: 85
Target:RUL


In [11]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=1,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

selected set: ['NIS_Steady_State_Nitrogen_Pressure_bar', 'CS_C2_PT20_bar', 'DS_D2_Nozzle_Pressure_bar', 'LP_M2_Pressure_bar', 'LP_M1_Pressure_bar', 'ES_PA_Pressure_bar', 'CS_C2_TIT10_celcius', 'DS_D1_Fuel_Coupling_Pressure_bar', 'DS_D1_Inlet_Pressure_bar', 'ES_PB_Pressure_bar', 'DS_D2_Nozzle_Temperature_celcius', 'DS_D2_Targeted_Pressure_bar', 'DS_D2_Fuel_Coupling_Temperature_celcius', 'DS_D2_Inlet_Pressure_bar', 'CS_C1_PT10_bar', 'DS_D1_Nozzle_Pressure_bar', 'Alarm_ID_915.0', 'LP_M1_Outflow_Temperature_celcius', 'LP_M1_Inflow_Temperature_celcius', 'Alarm_ID_817.0', 'Alarm_ID_925.0', 'Alarm_ID_115.0', 'Alarm_ID_1023410221.0', 'Alarm_ID_484.0', 'Alarm_ID_1444.0', 'Alarm_ID_1443.0', 'CS_C1_TIT400_celcius', 'NIS_After_Operation_Nitrogen_Pressure_bar', 'CS_C1_TIT10_celcius', 'Alarm_ID_1517.0', 'Alarm_ID_930.0', 'Alarm_ID_824.0', 'DS_D1_Fuel_Coupling_Temperature_celcius', 'CS_C2_PT400_bar', 'Alarm_ID_1357.0', 'Alarm_ID_877.0', 'Alarm_ID_1668.0', 'ES_PB_Total_Injection_Pressure_bar', 'Alarm_

In [ ]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=10,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

In [ ]:
fs_instance = ChronoEpilogi(
    X,
    target,
    phases="FBEV",
    target_type=target_type,
    start_with_univariate_autoregressive_model=False,
    default_max_lag=20,  
    variable_types=variable_types
)

fs_instance.fit()

print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)